**i. Perkenalan**

Milestone 3

Nama : Dianah Salsha Dilla

Batch : CODA-RMT-014

Program ini dibuat untuk melakukan evaluasi kemampuan penggunaan pipeline automation khususnya pada Phase 1

**ii. Data Loading**

Data yang digunakan berasal dari Kaggle dengan Judul Hajj and Umrah Crowd Management Dataset (2024) (referensi: https://www.kaggle.com/datasets/yarayahyahassan/hajj-and-umrah-crowd-management-dataset-2024)

In [ ]:
#Menginstall library Great Expectations ke dalam sistem Google Colab
!pip install -q great-expectations

In [ ]:
#a. Memanggil Libraries kedalam Notebook
#Mengolah dan memuat dataset dalam bentuk dataframe
import pandas as pd
#Mempersiapkan validasi data (7 expectations)
import great_expectations as ge
#Membuat konteks penyimpanan konfigurasi validasi
from great_expectations.data_context import FileDataContext
#Memastikan jalur file dataset sudah benar di sistem
import os

In [ ]:
#b. Mengimport data kedalam Notebook
df = pd.read_csv('P2M3_Dianah_Salsha_Dilla_data_raw.csv')
df

**iii. Data Cleaning**

**iii.1. Eksplorasi Data Sederhana**

In [ ]:
#a. Mencari informasi jumlah baris dan kolom
df.shape

In [ ]:
#b. Memunculkan tipe data
df.info()

In [ ]:
#c. Melihat statistik deskriptif
df.describe()

**iii.2. Missing Value Check & Handling**

In [ ]:
#a. Mengecek jumlah missing values per kolom
print(df.isnull().sum())

**iii.3. Duplicate Data Check & Handling**

In [ ]:
#a. Mengecek jumlah data duplikat
print(f'Jumlah duplikat: {df.duplicated().sum()}')

**iii.4. Inconsistency Data Check & Handling**

In [ ]:
#a. Melakukan pengecekan di daftar kategori unik pada kolom Incident_Type
print(df['Incident_Type'].unique())

In [ ]:
#b. Melakukan pengecekan daftar kategori unik pada kolom Age_Group
print(df['Age_Group'].unique())

In [ ]:
#c. Melakukan pengecekan daftar kategori unik pada kolom Activity_Type
print(df['Activity_Type'].unique())

In [ ]:
#d. Melakukan pengecekan daftar kategori unik pada kolom Weather_Conditions
print(df['Weather_Conditions'].unique())

**iii.5. Final Data Check**

In [ ]:
df.info()

**iv. Great Expectations**

**iv.1. Setup Data Context & Validator**

In [ ]:
#a. Membuat context otomatis di memori notebook
context = ge.get_context()

In [ ]:
#b. Membuat atau Mengambil Datasource
datasource_name = 'hajj_datasource' #Memberi nama unik pada datasource
try:
    datasource = context.data_sources.add_pandas(name=datasource_name) #Mencoba membuat datasource baru
except:
    datasource = context.data_sources.get(datasource_name) #Menggunakan nama datasource yang sudah ada jika sudah terdaftar

#c. Membuat Data Asset dari df
asset_name = 'hajj_asset' #memberi nama aset data
try:
    asset = datasource.add_dataframe_asset(name=asset_name) #mendaftarkan df sebagai aset baru
except:
    asset = datasource.get_asset(asset_name) #mengambil aset yang sudah ada jika sudah terdaftar

In [ ]:
# d. Memberi nama untuk aturan validasi
expectation_suite_name = 'hajj_expectation_suite'

# e. Membuat Expectation Suite
try:
    # Membuat suite baru di dalam context
    suite = context.suites.add(ge.ExpectationSuite(name=expectation_suite_name))
except:
    # Mengambil suite yang sudah ada jika cell dijalankan ulang
    suite = context.suites.get(name=expectation_suite_name)

# f. Membuat validator
validator = context.get_validator(
    batch_request = asset.build_batch_request(options={"dataframe": df}), # untuk memasukkan dataframe
    expectation_suite_name = expectation_suite_name
)

# g. Mengecek Validator
# Menampilkan 5 baris pertama
validator.head()

**iv.2. Great Expectations**

In [ ]:
#1. To be Unique
#a. Menambahkan kolom ID unik baru berdasarkan urutan baris agar validasi sukses
df['id_unit'] = range(1, len(df)+1)

#b. Memperbarui validator agar menyertakan kolom baru
validator = context.get_validator(
    batch_request = asset.build_batch_request(options={"dataframe": df}), # untuk memuat ulang dataframe
    expectation_suite_name = expectation_suite_name
)

#c. Menjalankan ekspektasi pertama
validator.expect_column_values_to_be_unique("id_unit")

In [ ]:
#2. To be Between (memastikan kecepatan jamaah berada di rantang normal 0 hingga 5 m/s agar data tetap logis
validator.expect_column_values_to_be_between("Movement_Speed", min_value=0, max_value=5)

In [ ]:
#3. To be in Set (Memastikan kategori kepadatan hanya berisi nilaii standar (Low, Medium, High)).
validator.expect_column_values_to_be_in_set("Crowd_Density", ["Low", "Medium", "High"])


In [ ]:
#4. To be in Type List (Memastikan kolom Latitude bertipe float)
validator.expect_column_values_to_be_of_type("Location_Lat", "float")

In [ ]:
#5. Data Not Null (Memastikan data aktivitas jamaah tidak ada yang kosong)
validator.expect_column_values_to_not_be_null("Activity_Type")

In [ ]:
#6. To Match Regex (Mengecek koordinat Location_Lat diawali dengan angka saja)
validator.expect_column_values_to_match_regex("Location_Lat", r"^[0-9].*")

In [ ]:
#7. Column Count to Equal (Memvalidasi apakah jumlah kolom sudah benar)
validator.expect_table_column_count_to_equal(7)